In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 09.1 - Overview, paths, and storage policy
# Purpose:
# Separate chromosome and plasmid sequence for all 176 blaTEM-1-only genomes
# using the same Platon accuracy-mode method validated in the previous project.
#
# Storage policy:
# - source assemblies remain unchanged in project storage;
# - Platon environment, database, copied input genome, and working output are
#   kept only in temporary Colab storage;
# - genomes are processed one at a time;
# - after each successful genome, only chromosome FASTA, plasmid FASTA,
#   Platon TSV/JSON, and compact progress records are retained in project storage;
# - temporary per-genome files are deleted immediately;
# - reruns verify saved outputs and skip completed genomes.
#
# This notebook performs chromosome/plasmid separation only.
# It does not generate variable sequence elements or test MIC associations.

from pathlib import Path
import json
import os
import shutil
import subprocess
import tarfile
import time
import traceback

import pandas as pd
from IPython.display import display
PROJECT_ROOT = _repo_root()

NOTEBOOK_DIR = (
    PROJECT_ROOT
    / "03_Notebooks"
    / "04_Genome_Comparison"
)

INTERMEDIATE_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "09_Platon_Chromosome_Plasmid_176"
)

RESULTS_TABLE_DIR = (
    PROJECT_ROOT
    / "05_Results"
    / "Tables"
)

SOURCE_MANIFEST = (
    RESULTS_TABLE_DIR
    / "08_downloaded_176_assembly_manifest.csv"
)

CHROMOSOME_DIR = (
    INTERMEDIATE_DIR
    / "chromosome_fasta"
)

PLASMID_DIR = (
    INTERMEDIATE_DIR
    / "plasmid_fasta"
)

REPORT_DIR = (
    INTERMEDIATE_DIR
    / "platon_reports"
)

FAILED_LOG_DIR = (
    INTERMEDIATE_DIR
    / "failed_logs"
)

PROGRESS_FILE = (
    RESULTS_TABLE_DIR
    / "09_platon_176_progress.csv"
)

FINAL_MANIFEST_FILE = (
    RESULTS_TABLE_DIR
    / "09_platon_176_chromosome_plasmid_manifest.csv"
)

CONTIG_SUMMARY_FILE = (
    RESULTS_TABLE_DIR
    / "09_platon_176_contig_classification_summary.csv"
)

for path in [
    PROJECT_ROOT,
    NOTEBOOK_DIR,
    RESULTS_TABLE_DIR,
]:
    assert path.exists(), f"Required path not found: {path}"

for path in [
    INTERMEDIATE_DIR,
    CHROMOSOME_DIR,
    PLASMID_DIR,
    REPORT_DIR,
    FAILED_LOG_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Notebook 09 - Platon chromosome/plasmid separation")
print("Project root:", PROJECT_ROOT)
print("Notebook folder:", NOTEBOOK_DIR)
print("Output root:", INTERMEDIATE_DIR)
print()
print("Storage policy:")
print("- Platon software/database: temporary Colab storage only")
print("- One genome copied to temporary storage at a time")
print("- Temporary genome/work files deleted after each genome")
print("- Original 176 source assemblies are not deleted")
print("- Completed genomes are verified and skipped on rerun")
print()
print("Transition: Cell 09.2 will verify the exact 176-genome input manifest.")


In [ ]:
#@title Cell 09.2 - Verify the exact 176 source assemblies
# Purpose:
# Load the frozen Notebook 08 download manifest and verify that all 176
# blaTEM-1-only genomes have one readable source FASTA and unique identifiers.

assert SOURCE_MANIFEST.exists(), (
    f"Source manifest not found: {SOURCE_MANIFEST}"
)

source_manifest = pd.read_csv(SOURCE_MANIFEST)

required_columns = {
    "biosample",
    "assembly_accession",
    "log2_mic",
    "assembly_fasta_path",
}

missing_columns = (
    required_columns
    - set(source_manifest.columns)
)

assert not missing_columns, (
    "Source manifest is missing required columns: "
    + ", ".join(sorted(missing_columns))
)

assert len(source_manifest) == 176, (
    f"Expected 176 rows, found {len(source_manifest)}."
)

assert source_manifest["biosample"].nunique() == 176, (
    "BioSample identifiers are not unique."
)

assert source_manifest["assembly_accession"].nunique() == 176, (
    "Assembly accessions are not unique."
)

source_manifest["assembly_fasta_path"] = (
    source_manifest["assembly_fasta_path"].astype(str)
)

source_manifest["source_fasta_exists"] = (
    source_manifest["assembly_fasta_path"]
    .map(lambda x: Path(x).is_file())
)

source_manifest["source_fasta_bytes"] = (
    source_manifest["assembly_fasta_path"]
    .map(
        lambda x: Path(x).stat().st_size
        if Path(x).is_file()
        else 0
    )
)

missing_source = source_manifest.loc[
    ~source_manifest["source_fasta_exists"]
    | (source_manifest["source_fasta_bytes"] <= 0)
].copy()

print(
    "Verified readable source FASTAs:",
    int(source_manifest["source_fasta_exists"].sum()),
    "/ 176",
)

print(
    "Total source FASTA size:",
    f"{source_manifest['source_fasta_bytes'].sum() / 1024**3:.3f} GiB",
)

if not missing_source.empty:
    display(
        missing_source[
            [
                "biosample",
                "assembly_accession",
                "assembly_fasta_path",
                "source_fasta_bytes",
            ]
        ]
    )

assert missing_source.empty, (
    "At least one source genome FASTA is missing or empty. "
    "Stop before running Platon."
)

display(
    source_manifest[
        [
            "biosample",
            "assembly_accession",
            "log2_mic",
            "assembly_fasta_path",
        ]
    ].head()
)

print("\nCell 09.2 complete.")
print(
    "Transition: Cell 09.3 will install the same isolated "
    "Platon environment used previously."
)


In [ ]:
#@title Cell 09.3 - Install Platon in temporary Colab storage
# Purpose:
# Reuse the previous project's isolated micromamba installation method.
# Nothing in this cell is stored in project storage.

MICROMAMBA_ROOT = Path(
    "/content/micromamba"
)

MICROMAMBA_BIN = (
    MICROMAMBA_ROOT
    / "bin"
    / "micromamba"
)

PLATON_ENV = Path(
    "/content/nb09_platon_env"
)

if not MICROMAMBA_BIN.exists():
    print("Installing micromamba in temporary Colab storage...")

    MICROMAMBA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    archive_path = Path(
        "/content/micromamba.tar.bz2"
    )

    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--show-error",
            "https://micro.mamba.pm/api/"
            "micromamba/linux-64/latest",
            "-o",
            str(archive_path),
        ],
        check=True,
    )

    with tarfile.open(
        archive_path,
        "r:bz2",
    ) as archive:
        member = archive.getmember(
            "bin/micromamba"
        )
        archive.extract(
            member,
            path=MICROMAMBA_ROOT,
            filter="data",
        )

    MICROMAMBA_BIN.chmod(0o755)

    if archive_path.exists():
        archive_path.unlink()

if not PLATON_ENV.exists():
    print("Installing Platon and dependencies...")

    subprocess.run(
        [
            str(MICROMAMBA_BIN),
            "create",
            "-y",
            "-p",
            str(PLATON_ENV),
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "platon",
        ],
        check=True,
    )
else:
    print("Existing temporary Platon environment found.")

def platon_env_run(
    *args,
    check=True,
    capture_output=False,
):
    return subprocess.run(
        [
            str(MICROMAMBA_BIN),
            "run",
            "-p",
            str(PLATON_ENV),
            *map(str, args),
        ],
        check=check,
        capture_output=capture_output,
        text=True,
    )

version_result = platon_env_run(
    "platon",
    "--version",
    capture_output=True,
)

print(
    "Platon version:",
    (
        version_result.stdout.strip()
        or version_result.stderr.strip()
    ),
)

print("\nCell 09.3 complete.")
print(
    "Transition: Cell 09.4 will prepare the Platon database "
    "in temporary Colab storage."
)


In [ ]:
#@title Cell 09.4 - Prepare the Platon database in temporary Colab storage
# Purpose:
# Use the same direct Platon database download used previously.
# The database is never copied to Google Drive.
#
# The compressed archive is deleted immediately after successful extraction
# to reduce temporary disk use.

PLATON_DB_ROOT = Path(
    "/content/nb09_platon_database"
)

PLATON_DB = (
    PLATON_DB_ROOT
    / "db"
)

PLATON_DB_ARCHIVE = (
    PLATON_DB_ROOT
    / "db.tar.gz"
)

PLATON_DB_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

total, used, free = shutil.disk_usage(
    "/content"
)

print(
    "Free Colab disk before database setup:",
    f"{free / 1024**3:.1f} GiB",
)

if free < 8 * 1024**3:
    raise RuntimeError(
        "Less than 8 GiB of free Colab disk remains. "
        "Do not download/extract the Platon database."
    )

if not PLATON_DB.exists():

    if not PLATON_DB_ARCHIVE.exists():
        print(
            "Downloading the Platon database "
            "to temporary Colab storage..."
        )

        subprocess.run(
            [
                "wget",
                "--show-progress",
                "-O",
                str(PLATON_DB_ARCHIVE),
                "https://zenodo.org/record/"
                "4066768/files/db.tar.gz",
            ],
            check=True,
        )

    print("Extracting the Platon database...")

    with tarfile.open(
        PLATON_DB_ARCHIVE,
        "r:gz",
    ) as archive:
        archive.extractall(
            PLATON_DB_ROOT,
            filter="data",
        )

if not PLATON_DB.exists():
    raise FileNotFoundError(
        "Platon database directory was not created."
    )

if PLATON_DB_ARCHIVE.exists():
    PLATON_DB_ARCHIVE.unlink()
    print(
        "Deleted compressed Platon database archive "
        "after extraction."
    )

total, used, free = shutil.disk_usage(
    "/content"
)

print("Platon database ready:", PLATON_DB)
print(
    "Free Colab disk after database setup:",
    f"{free / 1024**3:.1f} GiB",
)

print("\nCell 09.4 complete.")
print(
    "Transition: Cell 09.5 will define the restartable "
    "one-genome processing and QC functions."
)


In [ ]:
#@title Cell 09.5 - Define restartable one-genome Platon processing
# Purpose:
# Define functions used by the smoke test and full run.
#
# A genome is treated as complete only when the saved Drive outputs pass QC.
# Temporary input/output files are deleted in a finally block after each genome.
# Failed logs are retained because they are small and useful for diagnosis.

TEMP_WORK_ROOT = Path(
    "/content/nb09_platon_work"
)

TEMP_WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def fasta_stats(path):
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "bytes": 0,
            "records": 0,
            "bp": 0,
        }

    records = 0
    bp = 0

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:
        for line in handle:
            if line.startswith(">"):
                records += 1
            else:
                bp += len(line.strip())

    return {
        "exists": True,
        "bytes": path.stat().st_size,
        "records": records,
        "bp": bp,
    }


def output_paths(biosample):
    return {
        "chromosome": (
            CHROMOSOME_DIR
            / f"{biosample}.chromosome.fasta"
        ),
        "plasmid": (
            PLASMID_DIR
            / f"{biosample}.plasmid.fasta"
        ),
        "tsv": (
            REPORT_DIR
            / f"{biosample}.tsv"
        ),
        "json": (
            REPORT_DIR
            / f"{biosample}.json"
        ),
        "failed_log": (
            FAILED_LOG_DIR
            / f"{biosample}.log"
        ),
    }


def saved_outputs_valid(biosample):
    paths = output_paths(biosample)

    chromosome = fasta_stats(
        paths["chromosome"]
    )

    plasmid = fasta_stats(
        paths["plasmid"]
    )

    tsv_ok = (
        paths["tsv"].is_file()
        and paths["tsv"].stat().st_size > 0
    )

    json_ok = (
        paths["json"].is_file()
        and paths["json"].stat().st_size > 0
    )

    # A valid chromosome FASTA must contain sequence.
    # A plasmid FASTA may legitimately contain zero records.
    valid = (
        chromosome["exists"]
        and chromosome["records"] > 0
        and chromosome["bp"] > 0
        and plasmid["exists"]
        and tsv_ok
        and json_ok
    )

    return valid


def write_progress(rows):
    progress = pd.DataFrame(rows)

    temp_file = (
        RESULTS_TABLE_DIR
        / "09_platon_176_progress.tmp.csv"
    )

    progress.to_csv(
        temp_file,
        index=False,
    )

    os.replace(
        temp_file,
        PROGRESS_FILE,
    )


def process_one_genome(row):
    biosample = str(
        row["biosample"]
    )

    assembly_accession = str(
        row["assembly_accession"]
    )

    source_fasta = Path(
        row["assembly_fasta_path"]
    )

    destination = output_paths(
        biosample
    )

    if saved_outputs_valid(
        biosample
    ):
        chromosome = fasta_stats(
            destination["chromosome"]
        )
        plasmid = fasta_stats(
            destination["plasmid"]
        )

        return {
            "biosample": biosample,
            "assembly_accession": assembly_accession,
            "status": "complete_existing",
            "chromosome_bp": chromosome["bp"],
            "chromosome_contigs": chromosome["records"],
            "plasmid_bp": plasmid["bp"],
            "plasmid_contigs": plasmid["records"],
            "error": "",
        }

    work_dir = (
        TEMP_WORK_ROOT
        / biosample
    )

    if work_dir.exists():
        shutil.rmtree(
            work_dir
        )

    input_dir = (
        work_dir
        / "input"
    )

    output_dir = (
        work_dir
        / "platon_output"
    )

    input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_input = (
        input_dir
        / f"{biosample}.fna"
    )

    try:
        if not source_fasta.is_file():
            raise FileNotFoundError(
                f"Source FASTA missing: {source_fasta}"
            )

        shutil.copy2(
            source_fasta,
            temp_input,
        )

        result = platon_env_run(
            "platon",
            "--db",
            str(PLATON_DB),
            "--output",
            str(output_dir),
            "--prefix",
            biosample,
            "--mode",
            "accuracy",
            "--threads",
            "2",
            "--verbose",
            str(temp_input),
            check=False,
        )

        temp_chromosome = (
            output_dir
            / f"{biosample}.chromosome.fasta"
        )

        temp_plasmid = (
            output_dir
            / f"{biosample}.plasmid.fasta"
        )

        temp_tsv = (
            output_dir
            / f"{biosample}.tsv"
        )

        temp_json = (
            output_dir
            / f"{biosample}.json"
        )

        temp_log = (
            output_dir
            / f"{biosample}.log"
        )

        if result.returncode != 0:
            log_text = (
                temp_log.read_text(
                    errors="replace"
                )
                if temp_log.exists()
                else (
                    result.stderr
                    or result.stdout
                    or "No Platon log available."
                )
            )

            destination[
                "failed_log"
            ].write_text(
                log_text,
                encoding="utf-8",
                errors="replace",
            )

            raise RuntimeError(
                "Platon returned non-zero exit status. "
                "See saved failed log."
            )

        chromosome = fasta_stats(
            temp_chromosome
        )

        plasmid = fasta_stats(
            temp_plasmid
        )

        if (
            not chromosome["exists"]
            or chromosome["records"] == 0
            or chromosome["bp"] == 0
        ):
            raise RuntimeError(
                "Platon chromosome FASTA is missing or empty."
            )

        if not temp_plasmid.exists():
            raise RuntimeError(
                "Platon plasmid FASTA was not created."
            )

        if (
            not temp_tsv.is_file()
            or temp_tsv.stat().st_size == 0
        ):
            raise RuntimeError(
                "Platon TSV output is missing or empty."
            )

        if (
            not temp_json.is_file()
            or temp_json.stat().st_size == 0
        ):
            raise RuntimeError(
                "Platon JSON output is missing or empty."
            )

        # Copy only final retained outputs to project storage.
        shutil.copy2(
            temp_chromosome,
            destination["chromosome"],
        )

        shutil.copy2(
            temp_plasmid,
            destination["plasmid"],
        )

        shutil.copy2(
            temp_tsv,
            destination["tsv"],
        )

        shutil.copy2(
            temp_json,
            destination["json"],
        )

        if destination["failed_log"].exists():
            destination["failed_log"].unlink()

        if not saved_outputs_valid(
            biosample
        ):
            raise RuntimeError(
                "Saved Drive outputs failed post-copy QC."
            )

        return {
            "biosample": biosample,
            "assembly_accession": assembly_accession,
            "status": "complete_new",
            "chromosome_bp": chromosome["bp"],
            "chromosome_contigs": chromosome["records"],
            "plasmid_bp": plasmid["bp"],
            "plasmid_contigs": plasmid["records"],
            "error": "",
        }

    except Exception as exc:
        error_text = (
            f"{type(exc).__name__}: {exc}"
        )

        if not destination["failed_log"].exists():
            destination[
                "failed_log"
            ].write_text(
                traceback.format_exc(),
                encoding="utf-8",
                errors="replace",
            )

        return {
            "biosample": biosample,
            "assembly_accession": assembly_accession,
            "status": "failed",
            "chromosome_bp": 0,
            "chromosome_contigs": 0,
            "plasmid_bp": 0,
            "plasmid_contigs": 0,
            "error": error_text,
        }

    finally:
        # Critical storage-control step:
        # remove the copied input genome and all temporary Platon outputs.
        if work_dir.exists():
            shutil.rmtree(
                work_dir,
                ignore_errors=True,
            )


print(
    "Processing functions ready."
)

print(
    "Temporary per-genome work root:",
    TEMP_WORK_ROOT,
)

print(
    "Drive outputs will be retained under:",
    INTERMEDIATE_DIR,
)

print("\nCell 09.5 complete.")
print(
    "Transition: Cell 09.6 will process exactly one genome "
    "as a smoke test before the long 176-genome run."
)


In [ ]:
#@title Cell 09.6 - Smoke test exactly one genome
# Purpose:
# Process the first genome whose saved outputs are not already complete.
# This confirms the Platon command, output names, copying, QC, and cleanup
# before starting the long 176-genome run.

smoke_row = None

for _, row in source_manifest.iterrows():
    if not saved_outputs_valid(
        str(row["biosample"])
    ):
        smoke_row = row
        break

if smoke_row is None:
    print(
        "All 176 genomes already have valid saved outputs. "
        "No smoke test is required."
    )

else:
    smoke_biosample = str(
        smoke_row["biosample"]
    )

    print(
        "Smoke-test genome:",
        smoke_biosample,
    )

    before_free = shutil.disk_usage(
        "/content"
    ).free

    start_time = time.time()

    smoke_result = process_one_genome(
        smoke_row
    )

    elapsed_minutes = (
        time.time()
        - start_time
    ) / 60

    after_free = shutil.disk_usage(
        "/content"
    ).free

    display(
        pd.DataFrame(
            [smoke_result]
        )
    )

    print(
        "Elapsed:",
        f"{elapsed_minutes:.1f} minutes",
    )

    print(
        "Temporary Colab disk change after cleanup:",
        f"{(after_free - before_free) / 1024**2:.1f} MiB",
    )

    print(
        "Temporary genome work directory still exists:",
        (
            TEMP_WORK_ROOT
            / smoke_biosample
        ).exists(),
    )

    assert (
        smoke_result["status"]
        in {
            "complete_new",
            "complete_existing",
        }
    ), (
        "Smoke test failed. Stop here and review the "
        "saved failed log before running Cell 09.7."
    )

    assert not (
        TEMP_WORK_ROOT
        / smoke_biosample
    ).exists(), (
        "Temporary per-genome work directory was not deleted."
    )

print("\nCell 09.6 complete.")
print(
    "Transition: after reviewing this smoke-test output, "
    "Cell 09.7 can run the restartable 176-genome loop."
)


In [ ]:
#@title Cell 09.7 - Run the restartable 176-genome Platon separation
# Purpose:
# Process all 176 genomes sequentially.
#
# Restart behavior:
# - valid completed genomes are skipped;
# - incomplete/failed genomes are retried;
# - the progress table is saved after every genome;
# - a Colab interruption therefore does not discard completed work.
#
# Storage behavior:
# - one temporary genome/work directory at a time;
# - temporary files deleted immediately after each genome;
# - only final minimal outputs retained in project storage.

progress_rows = []

total_genomes = len(
    source_manifest
)

for position, (_, row) in enumerate(
    source_manifest.iterrows(),
    start=1,
):
    biosample = str(
        row["biosample"]
    )

    print(
        f"\n[{position}/{total_genomes}] "
        f"{biosample}",
        flush=True,
    )

    start_time = time.time()

    result = process_one_genome(
        row
    )

    result["position"] = position
    result["elapsed_minutes"] = (
        time.time()
        - start_time
    ) / 60

    progress_rows.append(
        result
    )

    # Save a full current-state progress table after every genome.
    # Existing completed outputs are re-verified each time this cell is rerun.
    write_progress(
        progress_rows
    )

    print(
        "Status:",
        result["status"],
        "| elapsed:",
        f"{result['elapsed_minutes']:.1f} min",
        flush=True,
    )

    if result["status"] == "failed":
        print(
            "Error:",
            result["error"],
            flush=True,
        )

progress = pd.DataFrame(
    progress_rows
)

print("\nRun summary:")

display(
    progress[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="n_genomes"
    )
)

failed = progress.loc[
    progress["status"] == "failed"
].copy()

if not failed.empty:
    print(
        "\nFailed genomes remain. "
        "Rerun Cell 09.7 after reviewing their small saved logs."
    )

    display(
        failed[
            [
                "position",
                "biosample",
                "assembly_accession",
                "error",
            ]
        ]
    )
else:
    print(
        "\nNo failed genomes in this run."
    )

print(
    "\nProgress file:",
    PROGRESS_FILE,
)

print(
    "\nTransition: Cell 09.8 will independently verify all "
    "176 saved outputs before this notebook is considered complete."
)


In [ ]:
#@title Cell 09.8 - Final verification and frozen chromosome/plasmid manifest
# Purpose:
# Independently verify all 176 retained outputs in project storage.
# The notebook is complete only if every genome passes.
#
# This cell also records compact per-genome chromosome/plasmid sequence totals.
# It does not delete the original Notebook 08 source assemblies.

verification_rows = []

for _, row in source_manifest.iterrows():
    biosample = str(
        row["biosample"]
    )

    paths = output_paths(
        biosample
    )

    chromosome = fasta_stats(
        paths["chromosome"]
    )

    plasmid = fasta_stats(
        paths["plasmid"]
    )

    tsv_exists = (
        paths["tsv"].is_file()
        and paths["tsv"].stat().st_size > 0
    )

    json_exists = (
        paths["json"].is_file()
        and paths["json"].stat().st_size > 0
    )

    complete = saved_outputs_valid(
        biosample
    )

    verification_rows.append(
        {
            "biosample": biosample,
            "assembly_accession": row[
                "assembly_accession"
            ],
            "log2_mic": row[
                "log2_mic"
            ],
            "source_assembly_fasta": row[
                "assembly_fasta_path"
            ],
            "chromosome_fasta": str(
                paths["chromosome"]
            ),
            "plasmid_fasta": str(
                paths["plasmid"]
            ),
            "platon_tsv": str(
                paths["tsv"]
            ),
            "platon_json": str(
                paths["json"]
            ),
            "chromosome_contigs": chromosome[
                "records"
            ],
            "chromosome_bp": chromosome[
                "bp"
            ],
            "plasmid_contigs": plasmid[
                "records"
            ],
            "plasmid_bp": plasmid[
                "bp"
            ],
            "tsv_present": tsv_exists,
            "json_present": json_exists,
            "complete": complete,
        }
    )

final_manifest = pd.DataFrame(
    verification_rows
)

final_manifest.to_csv(
    FINAL_MANIFEST_FILE,
    index=False,
)

contig_summary = (
    final_manifest[
        [
            "biosample",
            "assembly_accession",
            "chromosome_contigs",
            "chromosome_bp",
            "plasmid_contigs",
            "plasmid_bp",
            "complete",
        ]
    ]
    .copy()
)

contig_summary.to_csv(
    CONTIG_SUMMARY_FILE,
    index=False,
)

n_complete = int(
    final_manifest["complete"].sum()
)

print(
    "Verified complete Platon separations:",
    n_complete,
    "/ 176",
)

print(
    "Total retained chromosome sequence:",
    f"{final_manifest['chromosome_bp'].sum() / 1024**3:.3f} GiB",
)

print(
    "Total retained plasmid sequence:",
    f"{final_manifest['plasmid_bp'].sum() / 1024**3:.3f} GiB",
)

display(
    final_manifest[
        [
            "chromosome_contigs",
            "chromosome_bp",
            "plasmid_contigs",
            "plasmid_bp",
        ]
    ].describe()
)

incomplete = final_manifest.loc[
    ~final_manifest["complete"]
].copy()

if not incomplete.empty:
    print(
        "\nIncomplete genomes:"
    )

    display(
        incomplete[
            [
                "biosample",
                "assembly_accession",
                "chromosome_contigs",
                "chromosome_bp",
                "plasmid_contigs",
                "plasmid_bp",
                "tsv_present",
                "json_present",
            ]
        ]
    )

assert n_complete == 176, (
    "Not all 176 genomes have verified chromosome/plasmid outputs. "
    "Rerun Cell 09.7 before proceeding."
)

assert (
    final_manifest["biosample"].nunique()
    == 176
)

assert (
    final_manifest[
        "assembly_accession"
    ].nunique()
    == 176
)

print(
    "\nSaved final manifest:",
    FINAL_MANIFEST_FILE,
)

print(
    "Saved compact sequence summary:",
    CONTIG_SUMMARY_FILE,
)

print(
    "\nFINAL STATUS: 176/176 chromosome/plasmid separations verified."
)

print(
    "\nNotebook 09 ends here."
)

print(
    "Do not delete the original Notebook 08 source assemblies "
    "until this final result has been reviewed."
)

print(
    "Do not generate variable sequence elements until the "
    "chromosome-only FASTAs have been accepted for the next stage."
)
